# Ch02_Adult_Income

『실전! AI 프로젝트 완주의 기술』의 인쇄용 PDF에 수록된 코드에서 추출한 학습용 노트북입니다.

**주의:** 인쇄물에서 자동 추출된 초안입니다. 일부 조판 문자·줄바꿈·별도 데이터 경로가 원본 소스와 다를 수 있으므로 실행 전 확인하세요. 각 코드 셀 위에 인쇄본 페이지를 표시했습니다. 데이터 다운로드 및 패키지 설치는 필요한 경우 직접 진행해야 합니다. 출판 전 실행 검증본으로 간주하지 마세요.

**인쇄본 63쪽 — 코드**

In [1]:
from shap.datasets import adult
X, y = adult(display=True)
print(X.shape)
print(y.shape)

(32561, 12)
(32561,)


**인쇄본 63쪽 — 출력 예시 (실행 코드 아님)**

```text
(32561, 12)
(32561,)
```

**인쇄본 63쪽 — 코드** 

In [4]:
from sklearn.model_selection import train_test_split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.1,
    stratify=y,
    random_state=0
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=1/9,
    stratify=y_temp,
    random_state=0
)
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(26048, 12)
(3256, 12)
(3257, 12)


**인쇄본 64쪽 — 코드**

In [5]:
numerical_columns = [
    'Age',
    'Education-Num',
    'Capital Gain',
    'Capital Loss',
    'Hours per week'
]
categorical_columns = [
    'Workclass',
    'Marital Status',
    'Occupation',
    'Relationship',
    'Race',
    'Sex',
    'Country'
]

**인쇄본 65쪽 — 코드**

In [6]:
import pandas as pd
X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns
)
X_val = pd.get_dummies(
    X_val,
    columns=categorical_columns
)
X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns
)
X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

**인쇄본 65쪽 — 코드**

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)
X_val[numerical_columns] = scaler.transform(
    X_val[numerical_columns]
)
X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

**인쇄본 66쪽 — 코드**

In [8]:
X_train = X_train.values.astype(float)
X_val = X_val.values.astype(float)
X_test = X_test.values.astype(float)
y_train = y_train.astype(float)
y_val = y_val.astype(float)
y_test = y_test.astype(float)
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(26048, 91)
(3256, 91)
(3257, 91)


**인쇄본 66쪽 — 출력 예시 (실행 코드 아님)**

```text
(26048, 91)
(3256, 91)
(3257, 91)
```

**인쇄본 67쪽 — 코드**

In [9]:
from sklearn.linear_model import LogisticRegression
model_LR = LogisticRegression(max_iter=1000)
model_LR.fit(X_train, y_train)
val_acc_LR = model_LR.score(X_val, y_val)
print("validation accuracy:", val_acc_LR)

validation accuracy: 0.8473587223587223


**인쇄본 67쪽 — 출력 예시 (실행 코드 아님)** 

```text
validation accuracy: 0.8510
```

**인쇄본 67쪽 — 코드**

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

**인쇄본 68쪽 — 코드**

In [11]:
opt = Adam(
    learning_rate=0.01,
    beta_1=0.9,
    beta_2=0.999
)
model.compile(
    optimizer=opt,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

**인쇄본 68쪽 — 코드**

In [12]:
hist = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    verbose=0
)
val_result = model.evaluate(
    X_val,
    y_val,
    verbose=0
)
print(val_result)

[0.3634624183177948, 0.8485872149467468]


**인쇄본 68쪽 — 출력 예시**

```text
[0.3591, 0.8474]
```

**인쇄본 69쪽 — 코드**

In [13]:
from tensorflow.keras.regularizers import l2
model.add(
    Dense(
        128,
        activation='relu',
        kernel_regularizer=l2(0.01)
    )
)

**인쇄본 70쪽 — 코드**

In [14]:
from tensorflow.keras.initializers import HeNormal
model.add(
    Dense(
        128,
        activation='relu',
        kernel_initializer=HeNormal()
    )
)

**인쇄본 70쪽 — 코드**

In [15]:
from tensorflow.keras.layers import BatchNormalization, ReLU
model.add(Dense(128))
model.add(BatchNormalization())
model.add(ReLU())

**인쇄본 71쪽 — 코드**

In [16]:
from tensorflow.keras.layers import Dropout
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

**인쇄본 71쪽 — 코드**

In [17]:
from tensorflow.keras.callbacks import EarlyStopping
es_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

**인쇄본 71쪽 — 코드**

In [18]:
hist = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    callbacks=[es_callback],
    verbose=0
)

**인쇄본 72쪽 — 코드**

In [19]:
from tensorflow.keras.callbacks import LearningRateScheduler
def scheduler(epoch, lr):
    if epoch in [20, 40, 60]:
        return 0.1 * lr
    return lr
lr_callback = LearningRateScheduler(scheduler)

**인쇄본 72쪽 — 코드**

In [20]:
hist = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    callbacks=[es_callback, lr_callback],
    verbose=0
)

**인쇄본 73쪽 — 코드**

In [21]:
test_loss, test_acc = model.evaluate(
    X_test,
    y_test,
    verbose=0
)
print("test loss:", test_loss)
print("test accuracy:", test_acc)

test loss: 0.3598875403404236
test accuracy: 0.8483266830444336


**인쇄본 73쪽 — 출력 예시**

```text
test loss: 0.3854
test accuracy: 0.8440
```